In [24]:
import sys
import optuna
import numpy as np
import pandas as pd
from itertools import combinations

sys.path.append("..")
from feature_engineering import time_based_train_test_split, get_return, round_to_step, get_mask
from databricks_connector import get_table

pd.set_option('display.max_columns', None)
target_col = "btts"
odds_col: str = "goalNoGoal_quote_currentGG"


# features = [
#  'goalNoGoal_chance_goal',
#  'goalNoGoal_chance_goalHome',
#  'goalNoGoal_chance_goalAway',
#  'goalNoGoal_multigoal_m13',
#  'goalNoGoal_multigoal_m14',
#  'goalNoGoal_multigoal_m24',
#  'goalNoGoal_multigoal_m13Home',
#  'goalNoGoal_multigoal_m13Away',
#  'goalNoGoal_multigoal_m24Home',
#  'goalNoGoal_multigoal_m24Away',
#  'goalNoGoal_quote_realGG',
#  'goalNoGoal_quote_initialGG',
#  'goalNoGoal_quote_initialNG',
#  'goalNoGoal_quote_currentGG',
#  'goalNoGoal_quote_currentNG',
#  'goalNoGoal_quote_diffRealCurrGG',
#  'goalNoGoal_quote_diffRealCurrNG',
#  'goalNoGoal_quote_diffInitialCurrGG',
#  'goalNoGoal_quote_diffInitialCurrNG',
#  'goalNoGoal_comparison_affini',
#  'goalNoGoal_comparison_flashback',
#  'goalNoGoal_stats_avgGoalHome',
#  'goalNoGoal_stats_avgGoalTakenHome',
#  'goalNoGoal_stats_avgGoalAway',
#  'goalNoGoal_stats_avgGoalTakenAway',
#  'goalNoGoal_flashback_goal',
#  'goalNoGoal_flashback_m13',
#  'goalNoGoal_flashback_m24',
#  'goalNoGoal_flashback_m35',
#  'underOver_chance_over05HT',
#  'underOver_chance_over052HT',
#  'underOver_chance_over15HT',
#  'underOver_chance_over15',
#  'underOver_chance_over25',
#  'underOver_chance_over35',
#  'underOver_chance_over45',
#  'underOver_quote_realO',
#  'underOver_quote_initialU',
#  'underOver_quote_initialO',
#  'underOver_quote_currentU',
#  'underOver_quote_currentO',
#  'underOver_quote_diffRealCurrU',
#  'underOver_quote_diffRealCurrO',
#  'underOver_quote_diffInitialCurrU',
#  'underOver_quote_diffInitialCurrO',
#  'underOver_comparison_affini',
#  'underOver_comparison_flashback',
#  'underOver_flashback_under05HT',
#  'underOver_flashback_over05HT',
#  'underOver_flashback_under15',
#  'underOver_flashback_over15',
#  'underOver_flashback_under25',
#  'underOver_flashback_over25',
#  'underOver_flashback_under35',
#  'underOver_flashback_over35',
#  'evaluation_valScala',
#  'evaluation_valMetrica',
#  'chance1x2_quote_current1',
#  'chance1x2_quote_current2'
#  ]


features = ["underOver_chance_over15HT",
"chance1x2_quote_current1",
"chance1x2_quote_current2",
"goalNoGoal_quote_currentGG",
"goalNoGoal_chance_goal",
"goalNoGoal_flashback_goal",
"goalNoGoal_stats_avgGoalHome",
"goalNoGoal_stats_avgGoalTakenAway",
"goalNoGoal_stats_avgGoalTakenHome",
"goalNoGoal_stats_avgGoalAway"]

In [2]:
# TODO: far scegliere a Optuna se tenere o meno una variabile
# TODO: aggiungere early stopping
# TODO: aggiungere mean e std col dato aggregato settimanalmente
# TODO: aggiungere handling dei nan
# TODO: in alcuni casi ho che il valore minimo consigliato è uguale a al massimo, questo porta a maschere che annullano il df, capire come risolvere (forse considerare metriche sui df binnati)

In [3]:
# Load data
query = f"""
            SELECT *
            FROM bet_master_analytics.strategies.{target_col}_table
        """

df_loaded = get_table(query)
df_loaded = df_loaded.sort_values('time').reset_index(drop=True)

# Train/test split
df = df_loaded.copy()
df["return"] = df.apply(lambda x: get_return(strategy=x["btts"], odds=x["goalNoGoal_quote_currentGG"]), axis=1)

df_train, df_test = time_based_train_test_split(df=df, time_col="time", train_frac=0.8)

In [ ]:
# Define features and relative step
feature_bins_map = {
    # "underOver_chance_over15HT": 5, # 5,
    # "underOver_comparison_affini": 1000, #0.5, #0.1,
    # "underOver_comparison_flashback": 500, #0.5, #0.1,
    # "goalNoGoal_chance_goal": 5, # ,
    # "goalNoGoal_flashback_goal": 5, # 5,
    # "goalNoGoal_stats_avgGoalHome": 0.5, #0.1, 
    # "goalNoGoal_stats_avgGoalTakenAway": 0.5, #0.1, 
    # "goalNoGoal_stats_avgGoalTakenHome": 0.5, #0.1, 
    # "goalNoGoal_stats_avgGoalAway": 0.5, #0.1, 
    # "goalNoGoal_quote_currentGG": 0.2, #0.5, #0.1,
    "evaluation_valScala": None, # se la feature è categorica, mettere None
    "evaluation_valMetrica": None, # se la feature è categorica, mettere None

 }

# Create the binned dataframe
df_train_binned = df_train.copy()

    
for feat, step in feature_bins_map.items():
    df_train_binned[feat] = [round_to_step(x, step) for x in df_train_binned[feat]]

    if isinstance(step, int):
        df_train_binned[feat] = df_train_binned[feat].astype("Int64")


for feat in feature_bins_map.keys():
    print(feat +"\n")
    print(df_train_binned[feat].sort_values(ascending=True).unique())
    print("\n")

evaluation_valScala

['48727' '49657' '49867' '50827' '<NA>']


evaluation_valMetrica

['114521' '147930' '221599' '225586' '354214' '398418' '494608' '757193'
 '<NA>']




In [36]:
target_col = "return"
min_obs = 50



def objective(trial):
    mask = pd.Series(True, index=df_train_binned.index)

    for feat, step in feature_bins_map.items():

        s = df_train_binned[feat]
        non_null = s.dropna()

        if non_null.empty:
            continue

        # TODO: valutare rimozione    
        # include_missing = trial.suggest_categorical(
        #     f"{feat}_include_missing",
        #     [True, False]
        # )

        # TODO: vincolo per >=, <=, <,>, da studiare
        # use_min = trial.suggest_categorical(f"{feat}_use_min", [True, False])
        # use_max = trial.suggest_categorical(f"{feat}_use_max", [True, False])

        # # almeno un vincolo deve essere attivo
        # if not use_min and not use_max:
        #     raise optuna.TrialPruned()

        # ---------------------------------
        # CASO 1: variabile categorica
        # ---------------------------------
        if not step:
            categorical_set = [
                list(c) for r in range(1, len(s.unique()) + 1)
                for c in combinations(s.unique(), r)
            ]
            categorical_feat = trial.suggest_categorical(f"{feat}_cat", categorical_set)

        # ---------------------------------
        # CASO 2: numerica continua/intera
        # ---------------------------------
        else:
            if pd.api.types.is_integer_dtype(s):
                feat_min = trial.suggest_int(
                    f"{feat}_min",
                    int(non_null.min()),
                    int(non_null.max()),
                    step=step
                )
                feat_max = trial.suggest_int(
                    f"{feat}_max",
                    feat_min,
                    int(non_null.max()),
                    step=step
                )
            else:
                feat_min = trial.suggest_float(
                    f"{feat}_min",
                    float(non_null.min()),
                    float(non_null.max()),
                    step=step
                )
                feat_max = trial.suggest_float(
                    f"{feat}_max",
                    feat_min,
                    float(non_null.max()),
                    step=step
                )

        # costruzione maschera feature
        feat_mask = pd.Series(True, index=s.index)

        # Categorical filtering
        feat_mask &= s.isin(categorical_feat)

        # if use_min:
        #     feat_mask &= s >= feat_min

        # if use_max:
        #     feat_mask &= s <= feat_max

        # if include_missing:
        #     feat_mask = feat_mask | s.isna()
        # else:
        #     feat_mask = feat_mask & s.notna()

        mask &= feat_mask

    selected = df_train_binned.loc[mask]

    if len(selected) < min_obs:
        return -1e9

    mean_return = selected[target_col].mean()
    score = mean_return * np.sqrt(len(selected))

    return float(score)


study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=100)

print("Best params:", study.best_params)
print("Best score:", study.best_value)

[I 2026-03-28 10:39:09,329] A new study created in memory with name: no-name-854d1495-cf3e-4079-baed-10d3cfe153db
[I 2026-03-28 10:39:09,340] Trial 0 finished with value: -0.7809599539108283 and parameters: {'evaluation_valScala_cat': ['49867', '50827'], 'evaluation_valMetrica_cat': ['114521', '398418', '225586', '354214']}. Best is trial 0 with value: -0.7809599539108283.
[I 2026-03-28 10:39:09,346] Trial 1 finished with value: -1.6088953628428275 and parameters: {'evaluation_valScala_cat': ['50827', '48727'], 'evaluation_valMetrica_cat': ['<NA>', '114521', '398418', '221599', '225586', '354214', '757193']}. Best is trial 0 with value: -0.7809599539108283.
[I 2026-03-28 10:39:09,353] Trial 2 finished with value: -1.4230180725425128 and parameters: {'evaluation_valScala_cat': ['50827', '48727'], 'evaluation_valMetrica_cat': ['<NA>', '398418', '494608', '221599', '354214', '757193']}. Best is trial 0 with value: -0.7809599539108283.
[I 2026-03-28 10:39:09,360] Trial 3 finished with valu

Best params: {'evaluation_valScala_cat': ['<NA>'], 'evaluation_valMetrica_cat': ['<NA>', '494608', '757193']}
Best score: 0.8082861068444215


In [37]:
study.best_params

{'evaluation_valScala_cat': ['<NA>'],
 'evaluation_valMetrica_cat': ['<NA>', '494608', '757193']}

In [44]:
# Create the test binned dataframe
df_test_binned = df_test.copy()

for feat, step in feature_bins_map.items():
    df_test_binned[feat] = [round_to_step(x, step) for x in df_test_binned[feat]]

    if isinstance(step, int):
        df_test_binned[feat] = df_test_binned[feat].astype("Int64")

# Get the resulting dataframes
params_dict =study.best_params

def get_mask(df, params_dict):
    mask = pd.Series(True, index=df.index)

    features = set()
    for key in params_dict:
        for suffix in ["_cat"]: # "_include_missing", "_use_min", "_use_max", "_min_idx", "_max_idx", "_min", "_max", "_cat"
             if key.endswith(suffix):
                features.add(key[:-len(suffix)])
                break

    for feat in features:
        s = df[feat]

       
        # include_missing = params_dict.get(f"{feat}_include_missing", False)
        # use_min = params_dict.get(f"{feat}_use_min", True)
        # use_max = params_dict.get(f"{feat}_use_max", True)

        feat_mask = pd.Series(True, index=df.index)

        if f"{feat}_cat" in params_dict:
            feat_mask &= s.isin(params_dict[f"{feat}_cat"])

        # if f"{feat}_min" in params_dict and use_min:
        #     feat_mask &= s >= params_dict[f"{feat}_min"]

        # if f"{feat}_max" in params_dict and use_max:
        #     feat_mask &= s <= params_dict[f"{feat}_max"]

        # if f"{feat}_min_idx" in params_dict and use_min:
        #     feat_mask &= s >= params_dict[f"{feat}_min_idx"]

        # if f"{feat}_max_idx" in params_dict and use_max:
        #     feat_mask &= s <= params_dict[f"{feat}_max_idx"]

        # if include_missing:
        #     feat_mask = feat_mask | s.isna()
        # else:
        #     feat_mask = feat_mask & s.notna()

        mask &= feat_mask

    return mask


train_mask = get_mask(df_train_binned, params_dict)
test_mask = get_mask(df_test_binned, params_dict)

df_train_filtered = df_train_binned[train_mask]
df_test_filtered = df_test_binned[test_mask]

In [45]:
mean_roi = df_train_filtered["return"].mean()
right = df_train_filtered["btts"].sum()  
total = df_train_filtered.shape[0]

accuracy = np.round((right / total), 1)
print(f"Mean ROI: {mean_roi}")
print(f"Accuracy: {accuracy} ({right}/{total})")

Mean ROI: 0.02435967302452317
Accuracy: 0.6 (627/1101)


In [46]:
mean_roi = df_test_filtered["return"].mean()
right = df_test_filtered["btts"].sum()  
total = df_test_filtered.shape[0]

accuracy = np.round((right / total), 1)
print(f"Mean ROI: {mean_roi}")
print(f"Accuracy: {accuracy} ({right}/{total})")

Mean ROI: -0.016996587030716718
Accuracy: 0.5 (157/293)
